In [1]:
import serial
import time
from datetime import datetime

# =========================================================
# CẤU HÌNH KẾT NỐI
# =========================================================
PORT = "COM14"
BAUD_RATE = 19200

LOG_FILE = f"rockblock_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"


# =========================================================
# LOG
# =========================================================
def write_log(message):
    """Ghi message ra màn hình và file log."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_message = f"[{timestamp}] {message}"

    print(log_message)

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(log_message + "\n")


# =========================================================
# GỬI AT COMMAND
# =========================================================
def send_cmd(ser, cmd, wait=2):
    """Gửi lệnh AT và ghi command + response vào log."""

    write_log(f">>> {cmd}")

    # Xóa dữ liệu cũ trong buffer
    ser.reset_input_buffer()

    # Gửi command
    ser.write((cmd + "\r").encode("ascii"))

    # Chờ modem phản hồi
    time.sleep(wait)

    # Đọc response
    data = ser.read_all().decode("ascii", errors="ignore").strip()

    if data:
        write_log(f"<<< {data}")
    else:
        write_log("<<< [NO RESPONSE]")

    return data


# =========================================================
# MAIN
# =========================================================
ser = None

try:
    write_log("=" * 70)
    write_log("ROCKBLOCK / IRIDIUM MODEM TEST")
    write_log("=" * 70)

    write_log(f"Serial port : {PORT}")
    write_log(f"Baud rate   : {BAUD_RATE}")
    write_log(f"Log file    : {LOG_FILE}")

    # -----------------------------------------------------
    # 1. KẾT NỐI SERIAL
    # -----------------------------------------------------
    ser = serial.Serial(
        port=PORT,
        baudrate=BAUD_RATE,
        timeout=2
    )

    write_log(f"Connected: {PORT} @ {BAUD_RATE}")

    # -----------------------------------------------------
    # 2. CẤU HÌNH MODEM
    # -----------------------------------------------------
    write_log("")
    write_log("=== INITIAL MODEM CONFIGURATION ===")

    send_cmd(ser, "AT")
    send_cmd(ser, "ATE0")
    send_cmd(ser, "AT&K0")

    # -----------------------------------------------------
    # 3. THÔNG TIN MODEM
    # -----------------------------------------------------
    write_log("")
    write_log("=== MODEM INFORMATION ===")

    firmware = send_cmd(ser, "AT+CGMR")
    imei = send_cmd(ser, "AT+CGSN")
    sim = send_cmd(ser, "AT+CPIN?")

    # -----------------------------------------------------
    # 4. KIỂM TRA SIM
    # -----------------------------------------------------
    if "READY" in sim:
        write_log("[OK] SIM status: READY")
    else:
        write_log("[WARNING] SIM status is not READY")

    # -----------------------------------------------------
    # 5. KIỂM TRA IRIDIUM NETWORK
    # -----------------------------------------------------
    write_log("")
    write_log("=" * 70)
    write_log("BAT DAU KIEM TRA TIN HIEU VA DONG BO IRIDIUM")
    write_log("Yeu cau dat anten ngoai troi")
    write_log("=" * 70)

    checks = 10
    network_connected = False

    for i in range(checks):

        write_log("")
        write_log(f"----- Check {i + 1}/{checks} -----")

        # -------------------------------------------------
        # CSQ
        # -------------------------------------------------
        csq = send_cmd(
            ser,
            "AT+CSQ",
            wait=2
        )

        # -------------------------------------------------
        # MSSTM
        # -------------------------------------------------
        msstm = send_cmd(
            ser,
            "AT-MSSTM",
            wait=3
        )

        # -------------------------------------------------
        # KIỂM TRA NETWORK
        # -------------------------------------------------
        msstm_lower = msstm.lower()

        if "no network service" not in msstm_lower:

            write_log(
                "[OK] Modem da ket noi thanh cong "
                "voi mang Iridium va dong bo thoi gian!"
            )

            network_connected = True
            break

        else:
            write_log(
                "[INFO] Chua co network service. "
                "Tiep tuc cho modem tim mang..."
            )

        # -------------------------------------------------
        # WAIT
        # -------------------------------------------------
        if i < checks - 1:

            write_log("Waiting 5 seconds...")
            time.sleep(5)

    # -----------------------------------------------------
    # 6. KẾT QUẢ CUỐI
    # -----------------------------------------------------
    write_log("")
    write_log("=" * 70)
    write_log("KET QUA")
    write_log("=" * 70)

    if network_connected:
        write_log("[SUCCESS] Iridium network available.")
    else:
        write_log(
            "[FAIL] Khong ket noi duoc Iridium network "
            f"sau {checks} lan kiem tra."
        )

    write_log("")
    write_log(f"Log saved to: {LOG_FILE}")


# =========================================================
# ERROR
# =========================================================
except serial.SerialException as se:

    write_log(f"[SERIAL ERROR] {se}")

except Exception as e:

    write_log(f"[ERROR] {e}")


# =========================================================
# CLOSE SERIAL
# =========================================================
finally:

    if ser is not None and ser.is_open:

        ser.close()

        write_log("Serial closed.")

    write_log("=" * 70)
    write_log("TEST FINISHED")
    write_log("=" * 70)

[2026-10-02 07:28:51] ======================================================================
[2026-10-02 07:28:51] ROCKBLOCK / IRIDIUM MODEM TEST
[2026-10-02 07:28:51] ======================================================================
[2026-10-02 07:28:51] Serial port : COM14
[2026-10-02 07:28:51] Baud rate   : 19200
[2026-10-02 07:28:51] Log file    : rockblock_log_20261002_072851.txt
[2026-10-02 07:28:51] Connected: COM14 @ 19200
[2026-10-02 07:28:51] 
[2026-10-02 07:28:51] === INITIAL MODEM CONFIGURATION ===
[2026-10-02 07:28:51] >>> AT
[2026-10-02 07:28:53] <<< OK
[2026-10-02 07:28:53] >>> ATE0
[2026-10-02 07:28:55] <<< OK
[2026-10-02 07:28:55] >>> AT&K0
[2026-10-02 07:28:57] <<< OK
[2026-10-02 07:28:57] 
[2026-10-02 07:28:57] === MODEM INFORMATION ===
[2026-10-02 07:28:57] >>> AT+CGMR
[2026-10-02 07:28:59] <<< Call Processor Version: TA21004

Modem DSP Version: 1.7 svn: 2358

DBB Version: 0x0001 (ASIC)

RFA Version: 0x001a (GRFA1B)

NVM Version: KVS

Hardware Version: BOOT07d4